# FreshCart ELT on Snowflake (local VS Code version)
Mock API data → raw landing files → Snowflake **bronze** → SQL → **silver** (+ **quarantine**) → **gold**, then quality checks.

Based on the course's reference notebook (`freshcart_elt_simple_(1).ipynb`) and flowchart. What is different here:

| Part | Where it lives |
|---|---|
| SQL (`SQL_00_SETUP` … `SQL_40_GOLD`) | `elt/sql/*.sql` (loaded into the constants of the same name) |
| Python: `write_landing`, `load_file`, `quality_checks`, `run_pipeline` | `elt/freshcart.py` |
| Sample ("mock") data | `elt/mock_data.py`, copied from the reference notebook; there are no real APIs |
| Snowflake login | `.env` in the project root (git-ignored), never in the notebook |

Database: `FRESHCART_DB` (the reference uses `FRESHCART`). Run the cells from top to bottom.

## 1. Check the environment

In [1]:
# Dependencies come from requirements-elt.txt (installed once into the project's .venv).
import sys
import snowflake.connector
print("python", sys.version.split()[0], "| snowflake-connector", snowflake.connector.__version__)

python 3.12.6 | snowflake-connector 3.12.4


## 2. Log in to Snowflake
`fc.connect()` reads `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD` (and role, warehouse, database) from `.env`.
Only the role, warehouse and database are printed.

In [1]:
import freshcart as fc
fc.connect()

Connected: role=ACCOUNTADMIN warehouse=FRESHCART_WH database=FRESHCART_DB


## 3. The sample data (MOCK)
Eight raw order rows (6 from the website, 2 from the mobile app), plus customer tier changes, exchange rates and pincodes.
**This is mock data** taken from the reference notebook. The mess is deliberate: the SQL has to clean it.

In [3]:
import pandas as pd
print("web orders   :", len(fc.MOCK_WEB_ORDERS))
print("mobile orders:", len(fc.MOCK_MOBILE_ORDERS))
print("crm events   :", len(fc.MOCK_CRM_TIER_EVENTS))
print("fx replies   :", len(fc.MOCK_FX_RESPONSES))
pd.DataFrame(fc.MOCK_WEB_ORDERS)

web orders   : 6
mobile orders: 2
crm events   : 4
fx replies   : 1


,order_id,cust_email,cust_name,order_date,amount,currency,pincode,updated_at
0,ORD-1001,Priya.R@Gmail.com,priya r,2026-09-01,"1,250.00",INR,600041,2026-09-01 10:02:00
1,ORD-1003,meera@outlook.com,MEERA S,2026-09-02,None,INR,400050,2026-09-02 09:10:00
2,ORD-1002,Arjun.K@Yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,2026-09-01 11:15:00
3,ORD-1004,rahul.m@gmail.com,Rahul M,"Sep 3, 2026",-500,INR,110001,2026-09-03 12:00:00
4,ORD-1005,priya.r@gmail.com,Priya R.,2026-09-03,₹ 899,inr,600041,2026-09-03 16:45:00
5,ORD-1006,vikram@@mail,Vikram,2026-09-04,30,EUR,99999,2026-09-04 08:30:00


## 4. EXTRACT: save the data as raw files
`fc.write_landing(source, records, run_id)` writes one compressed `.json.gz` file per source (in `elt/landing/`). Nothing is cleaned.
Each record is wrapped in a small envelope:
```
{"source": "web", "run_id": "...", "extracted_at": "...", "data": { <the original record> }}
```

## 5. LOAD: copy a raw file into a bronze table
`fc.load_file(local_path, table, stage_dir)`: `PUT` uploads the file into the Snowflake stage `@bronze.landing`, `COPY INTO` reads it into a bronze table as raw JSON.

## 6. TRANSFORM: the SQL
Each script lives in `elt/sql/` and is loaded into a constant. Nothing runs yet; `run_pipeline()` sends them to Snowflake in this order.

In [4]:
for name in ["SQL_00_SETUP", "SQL_10_ORDERS_CLEAN", "SQL_20_ORDERS_DEDUP", "SQL_30_CUSTOMERS_SCD2",
             "SQL_31_FX_DAILY", "SQL_32_PINCODE_GEO", "SQL_40_GOLD"]:
    sql = getattr(fc, name)
    print(f"{name:24s} {len(sql.splitlines()):3d} lines")

SQL_00_SETUP              97 lines
SQL_10_ORDERS_CLEAN       73 lines
SQL_20_ORDERS_DEDUP       11 lines
SQL_30_CUSTOMERS_SCD2     22 lines
SQL_31_FX_DAILY           37 lines
SQL_32_PINCODE_GEO        25 lines
SQL_40_GOLD               47 lines


## 7. The pipeline
`fc.run_pipeline()` does the four stages in order: extract, load, transform, check.

## 8. Run it
First the setup script: it creates the schemas, stage, file format and empty tables (safe to run again).

In [5]:
fc.run_sql(fc.SQL_00_SETUP)
print("Setup done")

Setup done


**Before the first run**, confirm that bronze is empty. The B1 numbers (8 raw order rows) are only valid after exactly one run.

In [6]:
before = fc.query("SELECT COUNT(*) FROM bronze.orders_raw")[0][0]
print("bronze.orders_raw rows before the run:", before)
assert before == 0, "bronze is not empty: the B1 baseline needs a fresh start"

bronze.orders_raw rows before the run:

 0


Now the pipeline. Expected last line: **GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points**. "Share of rows rejected: 25%" is correct for this sample (2 bad rows out of 8).

In [7]:
run_info = fc.run_pipeline()

run_id: 20261112_040919
1. EXTRACT: sample data -> raw files
   web: 6 records -> elt\landing\web\web__20261112_040919.json.gz
   mobile: 2 records -> elt\landing\mobile\mobile__20261112_040919.json.gz
   crm: 4 records -> elt\landing\crm\crm__20261112_040919.json.gz
   fx: 1 records -> elt\landing\fx\fx__20261112_040919.json.gz
2. LOAD: raw files -> bronze tables


   6 rows -> bronze.orders_raw


   2 rows -> bronze.orders_raw


   4 rows -> bronze.customers_raw


   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold


   new pincodes to look up: ['600041', '560034', '400050']
   pincode: 3 records -> elt\landing\pincode\pincode__20261112_040919.json.gz


... (3 outputs left out)

   orders with no customer match          0   OK
   orders with an amount but no FX rate   0   OK


   orders with no city                    0   OK


   share of rows rejected                 25%


GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points


## 9. Look inside each layer
### Bronze: raw JSON exactly as received

In [8]:
fc.show("SELECT ingest_seq, source, raw FROM bronze.orders_raw ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,RAW
0,1,web,"{\n ""amount"": ""1,250.00"",\n ""currency"": ""INR..."
1,2,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
2,3,web,"{\n ""amount"": ""42.50"",\n ""currency"": ""USD"",\..."
3,4,web,"{\n ""amount"": ""-500"",\n ""currency"": ""INR"",\n..."
4,5,web,"{\n ""amount"": ""₹ 899"",\n ""currency"": ""inr"",\..."
5,6,web,"{\n ""amount"": 30,\n ""currency"": ""EUR"",\n ""c..."
6,7,mobile,"{\n ""customer"": {\n ""email"": ""arjun.k@yaho..."
7,8,mobile,"{\n ""customer"": {\n ""email"": ""Meera@Outloo..."


### Silver: cleaned rows (6)

In [9]:
fc.show("SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, amount_missing FROM silver.orders_clean ORDER BY ingest_seq")

,INGEST_SEQ,SOURCE,ORDER_ID,EMAIL,NAME,ORDER_DATE,AMOUNT,CURRENCY,PINCODE,AMOUNT_MISSING
0,1,web,ORD-1001,priya.r@gmail.com,Priya R,2026-09-01,1250.00,INR,600041,False
1,2,web,ORD-1003,meera@outlook.com,Meera S,2026-09-02,None,INR,400050,True
2,3,web,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
3,5,web,ORD-1005,priya.r@gmail.com,Priya R,2026-09-03,899.00,INR,600041,False
4,7,mobile,ORD-1002,arjun.k@yahoo.in,Arjun K,2026-09-01,42.50,USD,560034,False
5,8,mobile,ORD-1003,meera@outlook.com,Meera S,2026-09-02,2100.00,INR,400050,False


### Quarantine: rejected rows and why

In [10]:
fc.show("SELECT run_id, source, order_id, reasons FROM quarantine.orders ORDER BY order_id")

,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20261112_040919,web,ORD-1004,"[\n ""non_positive_amount""\n]"
1,20261112_040919,web,ORD-1006,"[\n ""bad_email"",\n ""bad_pincode""\n]"


### Silver: one row per order (4)

In [11]:
fc.show("SELECT order_id, ingest_seq, updated_at, amount FROM silver.orders_dedup ORDER BY order_id")

,ORDER_ID,INGEST_SEQ,UPDATED_AT,AMOUNT
0,ORD-1001,1,2026-09-01 10:02:00,1250.00
1,ORD-1002,7,2026-09-01 11:15:00,42.50
2,ORD-1003,8,2026-09-02 18:40:00,2100.00
3,ORD-1005,5,2026-09-03 16:45:00,899.00


### Dim: customer tier history (SCD2)

In [12]:
fc.show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

,CUST_ID,EMAIL,TIER,VALID_FROM,VALID_TO,IS_CURRENT
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10,2026-09-02,False
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03,9999-12-31,True
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15,9999-12-31,True
3,CUST-03,meera@outlook.com,Platinum,2022-11-01,9999-12-31,True


### Ref: daily FX rates, weekends filled

In [13]:
fc.show("SELECT * FROM ref.fx_daily WHERE currency = 'USD' ORDER BY rate_date LIMIT 10")

,CURRENCY,RATE_DATE,RATE_TO_INR,IS_CARRIED_FORWARD
0,USD,2026-09-01,83.2000,False
1,USD,2026-09-02,83.3500,False
2,USD,2026-09-03,83.1000,False
3,USD,2026-09-04,83.1000,True
4,USD,2026-09-05,83.1000,True
5,USD,2026-09-06,83.1000,True
6,USD,2026-09-07,83.1000,True
7,USD,2026-09-08,83.1000,True
8,USD,2026-09-09,83.1000,True
9,USD,2026-09-10,83.1000,True


### Ref: pincode geography

In [14]:
fc.show("SELECT * FROM ref.pincode_geo ORDER BY pincode")

,PINCODE,CITY,STATE,REGION,UPDATED_AT
0,400050,Mumbai,Maharashtra,West,2026-11-11 20:09:27.889000-08:00
1,560034,Bengaluru,Karnataka,South,2026-11-11 20:09:27.889000-08:00
2,600041,Chennai,Tamil Nadu,South,2026-11-11 20:09:27.889000-08:00


### Gold: the final table

In [15]:
fc.show("SELECT order_id, order_date, tier, amount, currency, fx_rate, amount_inr, loyalty_points, is_high_value, city, region FROM gold.fct_orders ORDER BY order_id")

,ORDER_ID,ORDER_DATE,TIER,AMOUNT,CURRENCY,FX_RATE,AMOUNT_INR,LOYALTY_POINTS,IS_HIGH_VALUE,CITY,REGION
0,ORD-1001,2026-09-01,Silver,1250.00,INR,1.0000,1250.00,12,False,Chennai,South
1,ORD-1002,2026-09-01,Silver,42.50,USD,83.2000,3536.00,35,True,Bengaluru,South
2,ORD-1003,2026-09-02,Platinum,2100.00,INR,1.0000,2100.00,63,True,Mumbai,West
3,ORD-1005,2026-09-03,Gold,899.00,INR,1.0000,899.00,16,False,Chennai,South


### Gold totals

In [16]:
fc.show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

,ORDERS,REVENUE_INR,POINTS
0,4,7785.00,126


## 10. B1 check: expected vs actual (after exactly one run)
Expected values come from the assignment table (B1) and the flowchart. `PASS` only if the actual number matches.

In [17]:
q = lambda sql: fc.query(sql)[0][0]
rejected_share = float(q("SELECT COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0) FROM silver.orders_checked"))
quarantined = sorted(r[0] for r in fc.query("SELECT order_id FROM quarantine.orders"))
gold_rev = q("SELECT SUM(amount_inr) FROM gold.fct_orders")
rows = [
    ("Raw order rows in bronze.orders_raw", 8, q("SELECT COUNT(*) FROM bronze.orders_raw")),
    ("  of which web", 6, q("SELECT COUNT(*) FROM bronze.orders_raw WHERE source = 'web'")),
    ("  of which mobile", 2, q("SELECT COUNT(*) FROM bronze.orders_raw WHERE source = 'mobile'")),
    ("Rows in bronze.customers_raw (tier changes)", 4, q("SELECT COUNT(*) FROM bronze.customers_raw")),
    ("Rows in bronze.fx_raw (FX replies)", 1, q("SELECT COUNT(*) FROM bronze.fx_raw")),
    ("Rows in bronze.pincode_raw (3 on the first run)", 3, q("SELECT COUNT(*) FROM bronze.pincode_raw")),
    ("Rows that pass cleaning (silver.orders_clean)", 6, q("SELECT COUNT(*) FROM silver.orders_clean")),
    ("Rows in quarantine.orders", 2, q("SELECT COUNT(*) FROM quarantine.orders")),
    ("Quarantined order ids", ["ORD-1004", "ORD-1006"], quarantined),
    ("Orders after deduplication (silver.orders_dedup)", 4, q("SELECT COUNT(*) FROM silver.orders_dedup")),
    ("Rows in dim.customers_scd2 (tiers)", 4, q("SELECT COUNT(*) FROM dim.customers_scd2")),
    ("Rows in ref.pincode_geo", 3, q("SELECT COUNT(*) FROM ref.pincode_geo")),
    ("Gold orders (gold.fct_orders)", 4, q("SELECT COUNT(*) FROM gold.fct_orders")),
    ("Gold revenue (INR)", "7785.00", f"{gold_rev:.2f}"),
    ("Gold loyalty points", 126, q("SELECT SUM(loyalty_points) FROM gold.fct_orders")),
    ("Share of rows rejected", "25%", f"{rejected_share:.0%}"),
]
result = pd.DataFrame([(n, e, a, "PASS" if str(e) == str(a) else "FAIL") for n, e, a in rows],
                      columns=["Check", "Expected", "Actual", "Result"])
print(f"{(result.Result == 'PASS').sum()} of {len(result)} checks PASS")
result

16 of 16 checks PASS


,Check,Expected,Actual,Result
0,Raw order rows in bronze.orders_raw,8,8,PASS
1,of which web,6,6,PASS
2,of which mobile,2,2,PASS
3,Rows in bronze.customers_raw (tier changes),4,4,PASS
4,Rows in bronze.fx_raw (FX replies),1,1,PASS
5,Rows in bronze.pincode_raw (3 on the first run),3,3,PASS
6,Rows that pass cleaning (silver.orders_clean),6,6,PASS
7,Rows in quarantine.orders,2,2,PASS
8,Quarantined order ids,"[ORD-1004, ORD-1006]","[ORD-1004, ORD-1006]",PASS
9,Orders after deduplication (silver.orders_dedup),4,4,PASS


## B3: add a validation rule: `unsupported_currency`
FreshCart only accepts orders in **INR** and **USD**. One line was added inside `ARRAY_CONSTRUCT_COMPACT(...)` in `elt/sql/10_orders_clean.sql` (`SQL_10_ORDERS_CLEAN`).
Any other currency (or a missing one) gets the reason `unsupported_currency`. The currency is already upper-cased by then, so `usd` and `inr` still pass.

In [2]:
# The reasons block of SQL_10_ORDERS_CLEAN, with the new line (marked B3)
lines = fc.SQL_10_ORDERS_CLEAN.splitlines()
start = next(i for i, l in enumerate(lines) if "ARRAY_CONSTRUCT_COMPACT" in l)
print("\n".join(lines[start:start + 9]))

  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL),   -- B3: only INR and USD are accepted
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
  ) AS reasons


### Before the re-run
The tables still hold what the **B1 run** produced with the *old* rule. `quarantine.orders` stores the reasons ORD-1006 was given when it was first rejected.

In [3]:
import pandas as pd
q = lambda sql: fc.query(sql)[0][0]

def snapshot():
    gold = fc.query("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")[0]
    return {
        "bronze.orders_raw rows": q("SELECT COUNT(*) FROM bronze.orders_raw"),
        "silver.orders_checked rows": q("SELECT COUNT(*) FROM silver.orders_checked"),
        "silver.orders_clean rows": q("SELECT COUNT(*) FROM silver.orders_clean"),
        "silver.orders_dedup rows": q("SELECT COUNT(*) FROM silver.orders_dedup"),
        "quarantine.orders rows": q("SELECT COUNT(*) FROM quarantine.orders"),
        "gold orders": gold[0], "gold revenue_inr": f"{gold[1]:.2f}", "gold loyalty points": gold[2],
    }

before = snapshot()
assert before["bronze.orders_raw rows"] == 8, "expected the B1 state (8 bronze order rows) before the B3 re-run"
print(before)
print("\nORD-1006 in quarantine.orders (BEFORE):")
display(fc.show("SELECT run_id, source, order_id, TO_JSON(reasons) AS reasons FROM quarantine.orders WHERE order_id = 'ORD-1006'"))
print("ORD-1006 in silver.orders_checked (BEFORE):")
display(fc.show("SELECT ingest_seq, source, order_id, currency, TO_JSON(reasons) AS reasons FROM silver.orders_checked WHERE order_id = 'ORD-1006'"))

{'bronze.orders_raw rows': 8, 'silver.orders_checked rows': 8, 'silver.orders_clean rows': 6, 'silver.orders_dedup rows': 4, 'quarantine.orders rows': 2, 'gold orders': 4, 'gold revenue_inr': '7785.00', 'gold loyalty points': 126}

ORD-1006 in quarantine.orders (BEFORE):


,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20261112_040919,web,ORD-1006,"[""bad_email"",""bad_pincode""]"


ORD-1006 in silver.orders_checked (BEFORE):


,INGEST_SEQ,SOURCE,ORDER_ID,CURRENCY,REASONS
0,6,web,ORD-1006,EUR,"[""bad_email"",""bad_pincode""]"


### Re-run the pipeline (with the new rule)

In [4]:
b3_info = fc.run_pipeline()

run_id: 20261112_042326
1. EXTRACT: sample data -> raw files
   web: 6 records -> elt\landing\web\web__20261112_042326.json.gz
   mobile: 2 records -> elt\landing\mobile\mobile__20261112_042326.json.gz
   crm: 4 records -> elt\landing\crm\crm__20261112_042326.json.gz
   fx: 1 records -> elt\landing\fx\fx__20261112_042326.json.gz
2. LOAD: raw files -> bronze tables


   6 rows -> bronze.orders_raw


   2 rows -> bronze.orders_raw


   4 rows -> bronze.customers_raw


   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold


   new pincodes to look up: []


4. CHECK


   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK


   orders with no customer match          0   OK


   orders with an amount but no FX rate   0   OK
   orders with no city                    0   OK


   share of rows rejected                 25%
GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points


### After the re-run

In [5]:
after = snapshot()
comparison = pd.DataFrame({"before": before, "after": after})
comparison["changed?"] = ["yes" if str(b) != str(a) else "no" for b, a in zip(comparison["before"], comparison["after"])]
comparison

,before,after,changed?
bronze.orders_raw rows,8,16,yes
silver.orders_checked rows,8,16,yes
silver.orders_clean rows,6,12,yes
silver.orders_dedup rows,4,4,no
quarantine.orders rows,2,2,no
gold orders,4,4,no
gold revenue_inr,7785.00,7785.00,no
gold loyalty points,126,126,no


In [6]:
print("ORD-1006 in silver.orders_checked (AFTER; rebuilt by the re-run from all bronze rows):")
display(fc.show("SELECT ingest_seq, source, order_id, currency, TO_JSON(reasons) AS reasons FROM silver.orders_checked WHERE order_id = 'ORD-1006' ORDER BY ingest_seq"))
print("ORD-1006 in quarantine.orders (AFTER):")
display(fc.show("SELECT run_id, source, order_id, TO_JSON(reasons) AS reasons FROM quarantine.orders WHERE order_id = 'ORD-1006'"))

ORD-1006 in silver.orders_checked (AFTER; rebuilt by the re-run from all bronze rows):


,INGEST_SEQ,SOURCE,ORDER_ID,CURRENCY,REASONS
0,6,web,ORD-1006,EUR,"[""bad_email"",""bad_pincode"",""unsupported_curren..."
1,14,web,ORD-1006,EUR,"[""bad_email"",""bad_pincode"",""unsupported_curren..."


ORD-1006 in quarantine.orders (AFTER):


,RUN_ID,SOURCE,ORDER_ID,REASONS
0,20261112_040919,web,ORD-1006,"[""bad_email"",""bad_pincode""]"


### Is only the right currency flagged?

In [7]:
fc.show("""
SELECT currency, COUNT(*) AS rows_in_orders_checked,
       COUNT_IF(ARRAY_CONTAINS('unsupported_currency'::VARIANT, reasons)) AS flagged_unsupported_currency
FROM silver.orders_checked GROUP BY currency ORDER BY currency
""")

,CURRENCY,ROWS_IN_ORDERS_CHECKED,FLAGGED_UNSUPPORTED_CURRENCY
0,EUR,2,2
1,INR,10,0
2,USD,4,0


### Did the re-run add a second quarantine row?
Bronze now holds **two copies** of every raw order (one per run), so `silver.orders_checked` has two copies of each rejected row, but `quarantine.orders` should still have one row per rejected record.
The `NOT EXISTS (...)` in `SQL_10_ORDERS_CLEAN` skips a rejected row when `quarantine.orders` already has the same `source`, `order_id` and `updated_at` (it does not look at the reasons).

In [8]:
print("copies of each rejected record in silver.orders_checked vs quarantine.orders:")
fc.show("""
SELECT c.source, c.order_id, c.updated_at,
       COUNT(*) AS copies_in_orders_checked,
       (SELECT COUNT(*) FROM quarantine.orders q
         WHERE q.source = c.source AND q.order_id = c.order_id
           AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at)) AS rows_in_quarantine
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
GROUP BY c.source, c.order_id, c.updated_at ORDER BY c.order_id
""")

copies of each rejected record in silver.orders_checked vs quarantine.orders:


,SOURCE,ORDER_ID,UPDATED_AT,COPIES_IN_ORDERS_CHECKED,ROWS_IN_QUARANTINE
0,web,ORD-1004,2026-09-03 12:00:00,2,1
1,web,ORD-1006,2026-09-04 08:30:00,2,1


## B4: customer tier change (SCD2) and idempotency
New CRM event added to `MOCK_CRM_TIER_EVENTS` in `elt/mock_data.py`: **CUST-02, arjun.k@yahoo.in, Gold, effective_from 2026-08-15**.
The predictions below were written and committed **before** this section was run (`report/answers.md`, "PRE-RUN PREDICTIONS").

In [2]:
import pandas as pd
q = lambda sql: fc.query(sql)[0][0]
print("events in MOCK_CRM_TIER_EVENTS:", len(fc.MOCK_CRM_TIER_EVENTS))
pd.DataFrame(fc.MOCK_CRM_TIER_EVENTS)

events in MOCK_CRM_TIER_EVENTS: 5


,cust_id,email,tier,effective_from
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15
3,CUST-03,meera@outlook.com,Platinum,2022-11-01
4,CUST-02,arjun.k@yahoo.in,Gold,2026-08-15


### Before the run: gold rows with `published_at` and `row_hash`

In [3]:
gold_before = fc.show("SELECT order_id, tier, multiplier, loyalty_points, row_hash, published_at FROM gold.fct_orders ORDER BY order_id")
bronze_before = {t: q(f"SELECT COUNT(*) FROM bronze.{t}") for t in ["orders_raw", "customers_raw", "fx_raw", "pincode_raw"]}
print("CUST-02 rows in dim.customers_scd2 BEFORE:", q("SELECT COUNT(*) FROM dim.customers_scd2 WHERE cust_id = 'CUST-02'"))
print("bronze row counts BEFORE:", bronze_before)
gold_before

CUST-02 rows in dim.customers_scd2 BEFORE: 1
bronze row counts BEFORE: {'orders_raw': 16, 'customers_raw': 8, 'fx_raw': 2, 'pincode_raw': 3}


,ORDER_ID,TIER,MULTIPLIER,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT
0,ORD-1001,Silver,1,12,4284627330879666653,2026-11-11 20:09:29.606000-08:00
1,ORD-1002,Silver,1,35,-7698738864412791922,2026-11-11 20:09:29.606000-08:00
2,ORD-1003,Platinum,3,63,-1632326326823866187,2026-11-11 20:09:29.606000-08:00
3,ORD-1005,Gold,2,16,-231880702048109571,2026-11-11 20:09:29.606000-08:00


### Run the pipeline once (full run: extract, load, transform, check)

In [4]:
b4_info = fc.run_pipeline()

run_id: 20261112_043003
1. EXTRACT: sample data -> raw files
   web: 6 records -> elt\landing\web\web__20261112_043003.json.gz
   mobile: 2 records -> elt\landing\mobile\mobile__20261112_043003.json.gz
   crm: 5 records -> elt\landing\crm\crm__20261112_043003.json.gz
   fx: 1 records -> elt\landing\fx\fx__20261112_043003.json.gz
2. LOAD: raw files -> bronze tables


   6 rows -> bronze.orders_raw


   2 rows -> bronze.orders_raw


   5 rows -> bronze.customers_raw


   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold


   new pincodes to look up: []


... (3 outputs left out)

   orders with an amount but no FX rate   0   OK


   orders with no city                    0   OK


   share of rows rejected                 25%


GOLD: 4 orders, revenue ₹7785.00, 161 loyalty points


### The five outcomes: committed prediction vs actual

In [2]:
import pandas as pd
q = lambda sql: fc.query(sql)[0][0]

# The B4 pipeline run above has run_id 20261112_043003 (= 04:30:03 UTC). Snowflake Time Travel gives the tables exactly as
# they were just before it, so the "before" side of every comparison is read from Snowflake itself, not retyped.
BEFORE_RUN = "2026-11-12 04:30:00 +0000"
at = f"AT(TIMESTAMP => '{BEFORE_RUN}'::TIMESTAMP_LTZ)"
gold_before = fc.show(f"SELECT order_id, tier, multiplier, loyalty_points, row_hash, published_at FROM gold.fct_orders {at} ORDER BY order_id")
bronze_before = {t: q(f"SELECT COUNT(*) FROM bronze.{t} {at}") for t in ["orders_raw", "customers_raw", "fx_raw", "pincode_raw"]}

gold_after = fc.show("SELECT order_id, tier, multiplier, loyalty_points, row_hash, published_at FROM gold.fct_orders ORDER BY order_id")
cmp_ = gold_before.merge(gold_after, on="ORDER_ID", suffixes=("_before", "_after"))
published_changed = int((cmp_["PUBLISHED_AT_before"] != cmp_["PUBLISHED_AT_after"]).sum())
hash_changed = int((cmp_["ROW_HASH_before"] != cmp_["ROW_HASH_after"]).sum())
ord1002 = gold_after[gold_after.ORDER_ID == "ORD-1002"].iloc[0]
actual = {
    "CUST-02 rows in dim.customers_scd2": q("SELECT COUNT(*) FROM dim.customers_scd2 WHERE cust_id = 'CUST-02'"),
    "Tier used for ORD-1002": ord1002.TIER,
    "Loyalty points for ORD-1002": int(ord1002.LOYALTY_POINTS),
    "Total loyalty points in gold": int(q("SELECT SUM(loyalty_points) FROM gold.fct_orders")),
    "Gold rows whose published_at changed": published_changed,
}
predicted = {  # copied from the committed PRE-RUN PREDICTIONS in report/answers.md (commit 72e290d)
    "CUST-02 rows in dim.customers_scd2": 2,
    "Tier used for ORD-1002": "Gold",
    "Loyalty points for ORD-1002": 70,
    "Total loyalty points in gold": 161,
    "Gold rows whose published_at changed": 1,
}
result = pd.DataFrame({"prediction": predicted, "actual": actual})
result["match?"] = ["YES" if str(p) == str(a) else "NO" for p, a in zip(result["prediction"], result["actual"])]
print("gold BEFORE the B4 run (Time Travel):")
display(gold_before)
print(f"{(result['match?'] == 'YES').sum()} of {len(result)} predictions match; rows with changed row_hash: {hash_changed}")
result

gold BEFORE the B4 run (Time Travel):


,ORDER_ID,TIER,MULTIPLIER,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT
0,ORD-1001,Silver,1,12,4284627330879666653,2026-11-11 20:09:29.606000-08:00
1,ORD-1002,Silver,1,35,-7698738864412791922,2026-11-11 20:09:29.606000-08:00
2,ORD-1003,Platinum,3,63,-1632326326823866187,2026-11-11 20:09:29.606000-08:00
3,ORD-1005,Gold,2,16,-231880702048109571,2026-11-11 20:09:29.606000-08:00


5 of 5 predictions match; rows with changed row_hash: 1


,prediction,actual,match?
CUST-02 rows in dim.customers_scd2,2,2,YES
Tier used for ORD-1002,Gold,Gold,YES
Loyalty points for ORD-1002,70,70,YES
Total loyalty points in gold,161,161,YES
Gold rows whose published_at changed,1,1,YES


In [3]:
print("CUST-02 in dim.customers_scd2 (AFTER):")
display(fc.show("SELECT cust_id, tier, valid_from, valid_to, is_current FROM dim.customers_scd2 WHERE cust_id = 'CUST-02' ORDER BY valid_from"))
print("Gold rows: before vs after (only ORD-1002 should differ):")
cmp_

CUST-02 in dim.customers_scd2 (AFTER):


,CUST_ID,TIER,VALID_FROM,VALID_TO,IS_CURRENT
0,CUST-02,Silver,2024-01-15,2026-08-14,False
1,CUST-02,Gold,2026-08-15,9999-12-31,True


Gold rows: before vs after (only ORD-1002 should differ):


,ORDER_ID,TIER_before,MULTIPLIER_before,LOYALTY_POINTS_before,ROW_HASH_before,PUBLISHED_AT_before,TIER_after,MULTIPLIER_after,LOYALTY_POINTS_after,ROW_HASH_after,PUBLISHED_AT_after
0,ORD-1001,Silver,1,12,4284627330879666653,2026-11-11 20:09:29.606000-08:00,Silver,1,12,4284627330879666653,2026-11-11 20:09:29.606000-08:00
1,ORD-1002,Silver,1,35,-7698738864412791922,2026-11-11 20:09:29.606000-08:00,Gold,2,70,3392257107350681567,2026-11-11 20:30:13.292000-08:00
2,ORD-1003,Platinum,3,63,-1632326326823866187,2026-11-11 20:09:29.606000-08:00,Platinum,3,63,-1632326326823866187,2026-11-11 20:09:29.606000-08:00
3,ORD-1005,Gold,2,16,-231880702048109571,2026-11-11 20:09:29.606000-08:00,Gold,2,16,-231880702048109571,2026-11-11 20:09:29.606000-08:00


### Idempotence: run the transformation a second time
Only the transformation is repeated (`SQL_30_CUSTOMERS_SCD2` rebuilds the dimension, `SQL_40_GOLD` rebuilds `silver.orders_enriched` and runs the `MERGE`). Nothing new is loaded.
The `MERGE` updates a row (and refreshes `published_at`) only when `row_hash` changed, so every `published_at` must stay as it was after the first B4 run.

In [4]:
gold_mid = fc.show("SELECT order_id, row_hash, published_at FROM gold.fct_orders ORDER BY order_id")
fc.run_sql(fc.SQL_30_CUSTOMERS_SCD2)
fc.run_sql(fc.SQL_40_GOLD)
gold_again = fc.show("SELECT order_id, row_hash, published_at FROM gold.fct_orders ORDER BY order_id")
idem = gold_mid.merge(gold_again, on="ORDER_ID", suffixes=("_run1", "_run2"))
idem["published_at_kept?"] = idem["PUBLISHED_AT_run1"] == idem["PUBLISHED_AT_run2"]
idem["row_hash_same?"] = idem["ROW_HASH_run1"] == idem["ROW_HASH_run2"]
print("rows whose published_at changed in the second pass:", int((~idem["published_at_kept?"]).sum()))
print("gold totals after the second pass:", fc.query("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")[0])
idem

rows whose published_at changed in the second pass: 0
gold totals after the second pass: (4, Decimal('7785.00'), 161)


,ORDER_ID,ROW_HASH_run1,PUBLISHED_AT_run1,ROW_HASH_run2,PUBLISHED_AT_run2,published_at_kept?,row_hash_same?
0,ORD-1001,4284627330879666653,2026-11-11 20:09:29.606000-08:00,4284627330879666653,2026-11-11 20:09:29.606000-08:00,True,True
1,ORD-1002,3392257107350681567,2026-11-11 20:30:13.292000-08:00,3392257107350681567,2026-11-11 20:30:13.292000-08:00,True,True
2,ORD-1003,-1632326326823866187,2026-11-11 20:09:29.606000-08:00,-1632326326823866187,2026-11-11 20:09:29.606000-08:00,True,True
3,ORD-1005,-231880702048109571,2026-11-11 20:09:29.606000-08:00,-231880702048109571,2026-11-11 20:09:29.606000-08:00,True,True


### Why does bronze grow by 8 order rows on every run?
Every run extracts the same 8 raw orders again, writes them to **new landing files** (the file name contains the new `run_id`), `PUT`s them to a new stage folder and `COPY INTO` appends them.
Bronze keeps every copy it is given (it has no deduplication; that is silver's job), so `bronze.orders_raw` grows by 6 web + 2 mobile = 8 rows per run, while silver (`orders_dedup`) and gold stay the same size.

In [5]:
print("bronze.orders_raw rows per run (each full pipeline run adds one group of 8):")
display(fc.show("SELECT run_id, COUNT(*) AS order_rows, COUNT_IF(source = 'web') AS web, COUNT_IF(source = 'mobile') AS mobile, COUNT(DISTINCT src_file) AS landing_files FROM bronze.orders_raw GROUP BY run_id ORDER BY run_id"))
bronze_after = {t: q(f"SELECT COUNT(*) FROM bronze.{t}") for t in ["orders_raw", "customers_raw", "fx_raw", "pincode_raw"]}
growth = pd.DataFrame({"before B4 run": bronze_before, "after B4 run": bronze_after})
growth["added by this run"] = growth["after B4 run"] - growth["before B4 run"]
print("this run added 8 order rows, 5 CRM rows (the 4 old events + the new one), 1 FX reply, and no pincodes (all already known):")
growth

bronze.orders_raw rows per run (each full pipeline run adds one group of 8):


,RUN_ID,ORDER_ROWS,WEB,MOBILE,LANDING_FILES
0,20261112_040919,8,6,2,2
1,20261112_042326,8,6,2,2
2,20261112_043003,8,6,2,2


this run added 8 order rows, 5 CRM rows (the 4 old events + the new one), 1 FX reply, and no pincodes (all already known):


,before B4 run,after B4 run,added by this run
orders_raw,16,24,8
customers_raw,8,13,5
fx_raw,2,3,1
pincode_raw,3,3,0


## 11. Run it again: nothing should double *(not run yet)*
Bronze grows by 8 order rows on every run, because it keeps every copy it is given. Quarantine and gold stay the same.
This section is run later (B3/B4), so the B1 evidence above is taken after exactly one run.

In [ ]:
fc.run_pipeline()
fc.show("""
SELECT (SELECT COUNT(*) FROM bronze.orders_raw)  AS bronze_rows,
       (SELECT COUNT(*) FROM quarantine.orders)  AS quarantine_rows,
       (SELECT COUNT(*) FROM gold.fct_orders)    AS gold_orders,
       (SELECT SUM(amount_inr) FROM gold.fct_orders) AS revenue_inr,
       (SELECT SUM(loyalty_points) FROM gold.fct_orders) AS points
""")

## 12. Clean up (optional)
Removes everything this notebook created in Snowflake. Uncomment to run.

In [ ]:
# fc.query("DROP DATABASE IF EXISTS FRESHCART_DB")
# fc.query("DROP WAREHOUSE IF EXISTS FRESHCART_WH")
fc.conn.close()